# P04 · Notebook 21 · Strong fine-tuning of BEATs on three tracks (stage 3 of the program)

**Three tracks, all at −6 dB, all four machine types:**

| Track | Input | What it stands for |
|---|---|---|
| **ARR** | MWF, the 8-microphone filter of notebook 19 | a plant with a microphone array |
| **MIC+** | microphone 0 after notebook 20's denoiser | a plant with one microphone and an installation recording |
| **MIC** | raw microphone 0 | the DCASE 2020 setting, and the fair comparison with published single-microphone numbers |

**The plan:**
- Start from notebook 16's fine-tuning recipe (LoRA + sub-center ArcFace).
- Then try seven ingredients, **one at a time**:
  - **EP:** scoring on the fine-tuned backbone (768) instead of the head (256).
  - **XC:** 12 extra machines from DCASE 2020.
  - **PL:** k-means pseudo-labels.
  - **TS:** twin noise swap.
  - **PD:** pseudo-clean distillation.
  - **SE:** a 3-seed ensemble.
  - **EF:** fusion with frozen ECHO.
- An ingredient is kept only if it raises pumps and fans by at least 0.01 and costs valves and slide rails at most 0.01.
- At the end, the final system faces frozen BEATs.

**Rules:** `results/mimii_beats/RULES_strong_finetune_written_before_running.md`, written before this run.

**It needs** in Google Drive:
- `P04_BEATs`;
- `P04_TwinFrontEnds` (notebook 19);
- `P04_DenoiseScore` (notebook 20).

Drive needs at least 2 GB free. Notebook 20's `P04_AudioCache` is no longer used: its array streams were stored as 16-bit integers, which clip.

**Runtime:**
- Choose **G4** (RTX PRO 6000): Runtime › Change runtime type. Then Runtime › Run all.
- Cell 9 times one training step before any download. Cell 11 then prints the estimate for the whole program. The expectation is about 4–6 hours on G4, for 38 trainings.
- Every training and its embeddings are saved to Drive › `P04_StrongFineTune` as soon as they finish. If Colab disconnects, run all again: finished trainings are skipped, and an unfinished one resumes from its last epoch.
- With background execution on, you can close the browser.

**At the end** `p04_n21_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, threading, math, gc
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR = MY / "P04_StrongFineTune", Path("/content/work"), MY / "P04_BEATs"
    N19, N20 = MY / "P04_TwinFrontEnds", MY / "P04_DenoiseScore"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_StrongFineTune")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", str(OUT)))
    N19, N20 = Path(os.environ.get("P04_N19_DIR", "P04_TwinFrontEnds")), Path(os.environ.get("P04_N20_DIR", "P04_DenoiseScore"))
for p in (OUT, WORK, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)
for p, who in ((N19, "notebook 19 (P04_TwinFrontEnds)"), (N20, "notebook 20 (P04_DenoiseScore)")):
    if not p.exists():
        raise SystemExit(f"missing {p}: the files of {who} must be in Drive")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, "· torch", torch.__version__, f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB"
      f" · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · The official BEATs code (the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 3 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 4 · Every number below was fixed in the rules before this notebook trained anything
RULES = "results/mimii_beats/RULES_strong_finetune_written_before_running.md"
PROGRAM = "results/mimii_beats/PROGRAM_full_written_before_running.md"
Z = "https://zenodo.org/records/3384388/files/"
MD5 = {"pump|6dB": "a09ba6060c10fc09cd4c8770213b0b9f", "pump|-6dB": "d20b783a0ff9c93d58f452f98c37b112",
       "fan|6dB": "0890f7d3c2fd8448634e69ff1d66dd47", "fan|-6dB": "f02ae808a58d84b6815b7ec38ff30879",
       "valve|6dB": "fe5fb7c337cd701b1d31dc641e621892", "valve|-6dB": "fdfaf185fea61b21e11952a070a4ada7",
       "slider|6dB": "838c2b3441858359c4704ef13a1b27ff", "slider|-6dB": "49913eda7d37f182cbf8ed5c984140e0"}
ZIPS = {}
for key, h in MD5.items():
    t_, lv_ = key.split("|"); nm = f"{lv_.replace('dB', '_dB')}_{t_}.zip"
    ZIPS[key] = (Z + nm + "?download=1", nm, h)
ZX = "https://zenodo.org/records/3727685/files/"                    # DCASE 2020 Task 2, additional training dataset
XMD5 = {"pump": "4e33ae9c0db5cc88437675f5e317caee", "fan": "567798854130c8019df8b664c095ed1e",
        "valve": "057cdd560caffb8305449541afd28c6d", "slider": "f84f186f4295fc23754abcdd1b0580a7"}
XZIPS = {t_: (ZX + f"eval_data_train_{t_}.zip?download=1", f"eval_data_train_{t_}.zip", h) for t_, h in XMD5.items()}
GB = {"pump": 15.9, "fan": 21.1, "valve": 15.0, "slider": 15.1}     # approximate size of each type's two MIMII zips
if TEST:
    _d = json.loads(os.environ["P04_TEST_DATA"])
    XZIPS = {t_: tuple(v) for t_, v in _d.pop("xc").items()}
    _d.pop("dcase", None)
    ZIPS = {k: tuple(v) for k, v in _d.items() if k.split("|")[1] in ("6dB", "-6dB")}
TYPES, MAIN, AUX = ["pump", "fan", "valve", "slider"], ["pump", "fan"], ["valve", "slider"]
LEVELS = ["6dB", "-6dB"]
SR, L, H = 16000, 1024, 512                                          # notebook 19's beamformer STFT (MWF weights are per bin of it)
STREAMS = ["CH0_m6", "CH0_6", "MWF_m6", "MWF_6"]                     # notebook 20's streams: microphone 0 and MWF₋₆, on x₋₆ and its 6 dB twin
SHOTS = [1, 3, 10]
SPLITS = ["A", "B", "B+1", "B+3", "B+10"]
FOLDS = {1: ["id_00", "id_02"], 2: ["id_04", "id_06"]}
TRACKS = ["ARR", "MIC+", "MIC"]
ALL_TRACKS = list(TRACKS)                                            # fixed order for random seeds
TRACK_INPUT = {"ARR": "MWF", "MIC+": "N2N-CH0", "MIC": "CH0"}
TWIN_DOMAIN = {"ARR": "MWF", "MIC+": "CH0"}                          # where TS mixes and PD's teacher is read
ORDER = ["EP", "XC", "PL", "TS", "PD", "SE", "EF"]
APPLIES = {"EP": TRACKS, "XC": TRACKS, "PL": TRACKS, "TS": ["ARR", "MIC+"], "PD": ["ARR", "MIC+"], "SE": TRACKS, "EF": TRACKS}
CFG = dict(lora_rank=16, lora_alpha=32, lora_dropout=0.1, emb_dim=256, subcenters=3, arc_scale=30.0, arc_margin=0.2,
           mixup_prob=0.5, mixup_beta=0.5, crop_s=4.0, lr_lora=2e-4, lr_head=1e-3, wd_head=1e-4,
           batch=32, epochs=10, warmup_epochs=1, clip_norm=5.0, xc_cap=600, pl_k=4, pl_starts=10, pl_seed=0,
           ts_prob=0.5, pd_balance_batches=8, se_seeds=[0, 1, 2], embed_batch=32)
KAPPA = {"6dB": 10 ** -0.6, "-6dB": 10 ** 0.6}                      # MIMII noise power relative to 0 dB
C_SWAP = math.sqrt((KAPPA["-6dB"] - KAPPA["6dB"]) / (KAPPA["-6dB"] + KAPPA["6dB"]))
BAR = 0.01
G0B_TOL, G0C_COS, G0C_CLIPS = 0.0005, 0.999, 32
GT_RANGE, GT_PAIRS = (0.95, 1.05), 200
DRIVE_MIN_GB, LDN_FLOOR = 2.0, 1e-6
FROZEN_REF = {"ARR": (0.9442, 0.8835), "MIC+": (0.8135, 0.8238), "MIC": (0.7633, 0.7767)}   # (development, confirmation): BEATs LDN1, notebook 20
PUBLISHED_M6 = {"pump": 0.8977, "fan": 0.8778, "valve": 0.9794, "slider": 0.9441}         # arXiv 2309.07500, Table 1, −6 dB, one microphone
T2_TARGET = {"pump": 0.9785, "fan": 0.9539}                                                # frozen BEATs + BF at 6 dB, B+10 (the program)
N16_CONTEXT = {"fine-tuned CH0": {"pump": (0.9553, 0.8281), "fan": (0.8173, 0.5762)},       # notebook 16, B1 on F, (A, B+10)
               "fine-tuned BF": {"pump": (0.9490, 0.8999), "fan": (0.8380, 0.6058)}}
N2N_CFG = dict(n_fft=512, hop=256, hidden=256, layers=2, epochs=60, batch=32, lr=1e-3, lr_end=1e-5, clip=5.0, crop_s=4.0,
               val_every=20, norm_crops=200, in_levels=["-6dB"], seed=0)                     # notebook 20's (the N2N cell needs it)
if TEST:
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    CFG.update(_c.get("cfg", {})); N2N_CFG.update(_c.get("n2n", {}))
    if os.environ.get("P04_TEST_TRACKS"):
        TRACKS = os.environ["P04_TEST_TRACKS"].split(",")
assert abs(C_SWAP - 0.93877) < 5e-5
print("rules:", RULES, "· C_SWAP", round(C_SWAP, 5), "·", json.dumps(CFG))
if DRIVE_FREE_GB < DRIVE_MIN_GB and not TEST:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. "
                     "Free some space in Google Drive (for example, the old audio cache P04_AudioCache, 5.4 GB), then run again.")

In [ ]:
# 5 · Downloads (MD5 against the Zenodo records), a background download of the next type, readers, notebook 11's splits
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


PREFETCH = {}


def prefetch(t):
    # download the next machine type while this one trains its denoisers (only if the disk has room)
    if t is None or t in PREFETCH or (OUT / f"done_{t}.json").exists():
        return
    if shutil.disk_usage(WORK).free / 1e9 < GB.get(t, 0) * 1.05 + 8:
        print(f"  (not enough free disk to download {t} in the background; it will download when its turn comes)")
        return

    def job():
        for lv in LEVELS:
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} {lv} failed: {repr(e)[:150]}; it will be retried)", flush=True)
    th = threading.Thread(target=job, daemon=True); th.start(); PREFETCH[t] = th
    print(f"  background download of {t} started")


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]



shutil.rmtree(WORK / "stores21", ignore_errors=True)               # the first attempt's 16-bit streams (they clipped; see the rules' amendment)
STORE = WORK / "stores21b"; STORE.mkdir(exist_ok=True)
FLOAT_STREAMS = {"MWF_m6", "MWF_6"}                                  # processed signals are kept in float16, never clipped


def ready(t):
    # the type's streams are in this session's local store
    return (STORE / f"{t}.json").exists()


def prefetch(t):
    # download the next type's two zips in the background while this one is prepared (only if the disk has room)
    if t is None or t in PREFETCH or ready(t):
        return
    if shutil.disk_usage(WORK).free / 1e9 < GB.get(t, 0) * 1.05 + 8:
        print(f"  (not enough free disk to download {t} in the background; it will download when its turn comes)")
        return

    def job():
        for lv in LEVELS:
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} {lv} failed: {repr(e)[:150]}; it will be retried)", flush=True)
    th = threading.Thread(target=job, daemon=True); th.start(); PREFETCH[t] = th
    print(f"  background download of {t} started")


def open2(t):
    return {lv: zipfile.ZipFile(get_file(ZIPS[f"{t}|{lv}"])) for lv in LEVELS}


def read2(zfs, name):
    # one clip at 6 dB and at −6 dB (its twins): level → (samples, 8) float32
    out = {}
    for lv, zf in zfs.items():
        x, sr = read_all(zf.read(name))
        assert sr == SR and x.ndim == 2 and x.shape[1] == 8, (lv, name, sr, x.shape)
        if not TEST:
            assert x.shape[0] == 10 * SR, (lv, name, x.shape)
        out[lv] = x
    return out


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res

In [ ]:
# 6 · The STFT and beamforming of notebook 19 (MWF is applied with its saved weights), and a reconstruction check
SQH = torch.sqrt(torch.hann_window(L, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft(x):
    # (channels, samples) float32 tensor → (channels, bins, frames) complex64; centred, zero padding, square-root Hann
    return torch.stft(x, n_fft=L, hop_length=H, window=SQH, center=True, pad_mode="constant", return_complex=True)


def istft(Y, n):
    return torch.istft(Y, n_fft=L, hop_length=H, window=SQH, center=True, length=n)


def to_dev(x):
    # (samples, channels) numpy → (channels, samples) tensor on the device
    return torch.from_numpy(np.ascontiguousarray(x.T)).to(DEVICE)


def beam(w, X):
    # w: (bins, 8) complex64 tensor, X: (8, bins, frames) → wᴴx: (bins, frames)
    return torch.einsum("km,mkt->kt", w.conj(), X)



_x = (np.random.default_rng(0).standard_normal((3 * SR, 8)) * 0.1).astype(np.float32)
_pr = float(np.abs(istft(stft(to_dev(_x)), len(_x)).cpu().numpy().T - _x).max())
print(f"STFT reconstruction error {_pr:.1e}")
assert _pr < 1e-4

In [ ]:
# 7 · Noise2Noise post-filter (notebook 19's cell, unchanged; it applies notebook 20's denoisers): the network, the training loop, inference, and a smoke test on synthetic twins
NF, NH = N2N_CFG["n_fft"], N2N_CFG["hop"]
NB = NF // 2 + 1
SQH2 = torch.sqrt(torch.hann_window(NF, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft2(x):
    return torch.stft(x, n_fft=NF, hop_length=NH, window=SQH2, center=True, pad_mode="constant", return_complex=True)


def istft2(Y, n):
    return torch.istft(Y, n_fft=NF, hop_length=NH, window=SQH2, center=True, length=n)


class N2NNet(nn.Module):
    # log-power per bin → Linear → ReLU → 2-layer bidirectional GRU → Linear → sigmoid: a real mask per bin and frame
    def __init__(self):
        super().__init__()
        self.register_buffer("mu", torch.zeros(NB)); self.register_buffer("sd", torch.ones(NB))
        self.inp = nn.Linear(NB, N2N_CFG["hidden"])
        self.rnn = nn.GRU(N2N_CFG["hidden"], N2N_CFG["hidden"], num_layers=N2N_CFG["layers"], batch_first=True, bidirectional=True)
        self.out = nn.Linear(2 * N2N_CFG["hidden"], NB)

    def forward(self, Y):
        f = torch.log10(Y.real ** 2 + Y.imag ** 2 + 1e-10).transpose(1, 2)
        z, _ = self.rnn(torch.relu(self.inp((f - self.mu) / self.sd)))
        return torch.sigmoid(self.out(z)).transpose(1, 2)


def n2n_loss(M, Y, T):
    # per sample Σ|M·Y − T|² / Σ|Y|² (the weight depends on the input only, so the Noise2Noise optimum is unchanged)
    num = ((M * Y - T).abs() ** 2).sum((1, 2)); den = (Y.abs() ** 2).sum((1, 2)).clamp_min(1e-12)
    return (num / den).mean()


def val_loss(net, vx, vt):
    net.eval(); tot = idl = 0.0
    with torch.no_grad():
        for i in range(0, len(vx), 32):
            Y, T = stft2(vx[i:i + 32]), stft2(vt[i:i + 32]); M = net(Y)
            tot += float(n2n_loss(M, Y, T)) * len(Y); idl += float(n2n_loss(torch.ones_like(M), Y, T)) * len(Y)
    net.train()
    return tot / len(vx), idl / len(vx)


def fit(get, n_pairs, norm_x, vx, vt, seed, epochs=None):
    # get(index array, rng) → (inputs, targets) float32 tensors on the device; each epoch is one pass over a fresh permutation
    epochs = epochs or N2N_CFG["epochs"]
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = N2NNet().to(DEVICE)
    with torch.no_grad():
        Fx = torch.cat([torch.log10(stft2(c).abs() ** 2 + 1e-10).permute(0, 2, 1).reshape(-1, NB) for c in norm_x])
        net.mu.copy_(Fx.mean(0)); net.sd.copy_(Fx.std(0).clamp_min(1e-3))
    B = N2N_CFG["batch"]; steps = max(1, math.ceil(n_pairs / B))
    opt = torch.optim.Adam(net.parameters(), lr=N2N_CFG["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps * epochs, eta_min=N2N_CFG["lr_end"])
    v0, vi0 = val_loss(net, vx, vt)
    log = [{"epoch": 0, "train": None, "val": round(v0, 6), "val_identity": round(vi0, 6), "seconds": 0.0}]
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(n_pairs); run = 0.0
        for b in range(steps):
            x, tg = get(order[b * B:(b + 1) * B], rng)
            Y, T = stft2(x), stft2(tg)
            loss = n2n_loss(net(Y), Y, T)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), N2N_CFG["clip"]); opt.step(); sched.step()
            run += float(loss.detach())
        v, vi = val_loss(net, vx, vt)
        log.append({"epoch": ep + 1, "train": round(run / steps, 6), "val": round(v, 6), "val_identity": round(vi, 6), "seconds": round(time.time() - t0, 1)})
    return net, log


@torch.no_grad()
def denoise(net, x):
    # x: (batch, samples) float32 tensor → the post-filtered signal
    Y = stft2(x)
    return istft2(net(Y) * Y, x.shape[1])


# smoke test: harmonic sounds with independent noise in input and target; the denoiser must beat doing nothing on held-out clips
rng = np.random.default_rng(1)
ns, nc = 16000, 96
tt = torch.arange(ns, dtype=torch.float32) / SR
clean = torch.stack([sum(torch.sin(2 * math.pi * float(f0) * k * tt + float(rng.uniform(0, 6))) / k for k in range(1, 9))
                     for f0 in rng.uniform(80, 400, nc)]).to(DEVICE)
nz = lambda g: torch.from_numpy(rng.standard_normal((nc, ns)).astype(np.float32)).to(DEVICE) * g
xin, xtg = clean + nz(2.0), clean + nz(1.0)
net_s, log_s = fit(lambda idx, r: (xin[idx], xtg[idx]), 80, [xin[:16]], xin[80:], xtg[80:], seed=0, epochs=15 if not TEST else 6)
v0, v, vi = log_s[0]["val"], log_s[-1]["val"], log_s[-1]["val_identity"]
print(f"N2N smoke test: validation loss {v0:.3f} untrained → {v:.3f} after {len(log_s) - 1} epochs · {vi:.3f} for doing nothing")
assert v < vi and v < v0, "smoke test: the Noise2Noise denoiser does not learn"
del net_s

In [ ]:
# 8 · Model pieces (notebook 2's code): our forward pass, LoRA, the head and the sub-center ArcFace layer
def encode(x, amp):
    """Waveforms (B, N) in [-1, 1] → mean over time of the last layer (B, 768), float32.
    Same steps as BEATs.extract_features without padding; the filter banks are always computed in float32."""
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # (B, frames, 128)
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float().mean(1)


class LoRALinear(nn.Module):
    """y = W x + (α / r) · B A x. W stays frozen; B starts at zero, so the model starts exactly as the frozen one."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(base.out_features, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class Head(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
        self.bn = nn.BatchNorm1d(d_out)

    def forward(self, x):
        return self.bn(self.lin(x))


class SubCenterArcFace(nn.Module):
    """Cosine to the closest of k centres per class; additive angular margin m on the true class; scale s."""
    def __init__(self, d, n_classes, k, s, m):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes * k, d))
        nn.init.xavier_uniform_(self.W)
        self.n, self.k, self.s, self.m = n_classes, k, s, m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def cosine(self, e):
        c = F.linear(F.normalize(e.float(), dim=1), F.normalize(self.W.float(), dim=1))
        return c.view(-1, self.n, self.k).max(dim=2).values

    def loss(self, cos, y):
        sin = torch.sqrt((1.0 - cos * cos).clamp(0.0, 1.0))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)      # keeps the target logit monotonic near θ = π
        one = F.one_hot(y, self.n).to(cos.dtype)
        return F.cross_entropy(self.s * (one * phi + (1.0 - one) * cos), y)


def equal_length_batches(clips, idx, batch):
    """Consecutive clips of the same length, at most `batch` per group (the same grouping as notebook 1)."""
    buf = []
    for i in idx:
        if buf and len(clips[i]) != len(clips[buf[0]]):
            yield buf; buf = []
        buf.append(i)
        if len(buf) == batch:
            yield buf; buf = []
    if buf:
        yield buf


def as_float(clips, ids):
    return torch.from_numpy(np.stack([clips[i] for i in ids]).astype(np.float32) / 32768.0).to(DEVICE)

In [ ]:
# 9 · LoRA on the query and value projections of all 12 layers, with one switch (PD's frozen teacher runs with it off); checks; a timed step
for p_ in model.parameters():
    p_.requires_grad_(False)                                          # the original weights never change


class LoRASwitch(LoRALinear):
    on = True                                                         # one switch for every block

    def forward(self, x):
        y = self.base(x)
        return y + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale if LoRASwitch.on else y


def tokens(x, amp):
    # waveforms (B, N) in [-1, 1] → last-layer tokens (B, T, 768), float32: notebook 2's encode without the final mean
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float()


rng0 = np.random.default_rng(0)
CROP = int(CFG["crop_s"] * SR)
x4 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (4, CROP)).astype(np.float32)).to(DEVICE)
with torch.no_grad():
    Z4 = tokens(x4, amp=False).mean(1)
    ref = model.extract_features(x4)[0].mean(1)
    gap_e = float((encode(x4, amp=False) - Z4).abs().max())
gap = float((Z4 - ref).abs().max())
print(f"our forward pass against BEATs.extract_features: largest difference {gap:.2e} (notebook 2's encode: {gap_e:.1e})")
assert gap < 1e-3 and gap_e < 1e-5, "stop: our forward pass does not reproduce BEATs"
LORA = []
for layer in model.encoder.layers:
    att = layer.self_attn
    for name in ("q_proj", "v_proj"):
        w = LoRASwitch(getattr(att, name), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"])
        setattr(att, name, w); LORA.append(w)
with torch.no_grad():
    gap = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
print(len(LORA), "LoRA blocks · with LoRA at zero, largest difference from frozen BEATs:", f"{gap:.2e}")
assert gap < 1e-5, "stop: adding LoRA changed the model before training"


def lora_params():
    return [p for w in LORA for p in (w.A, w.B)]


def fresh_adapters(seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen BEATs
    torch.manual_seed(seed)
    for w in LORA:
        w.reset()
    head = Head(768, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


head_t, arc_t = fresh_adapters(12345, 8)
opt_t = torch.optim.AdamW(lora_params() + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
model.train(); head_t.train(); arc_t.train(); times = []
for k in range(3):
    xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
    yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    loss = arc_t.loss(arc_t.cosine(head_t(tokens(xb, amp=AMP).mean(1))), yb)
    opt_t.zero_grad(set_to_none=True); loss.backward()
    if k == 0:
        assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in LORA if w.B.grad is not None) > 0, "stop: no gradient reaches LoRA"
    opt_t.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
model.eval()
with torch.no_grad():
    moved = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = False
    off = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = True
print(f"after 3 steps: with LoRA on the output moved by {moved:.2e}; with the switch off it differs from frozen BEATs by {off:.2e}")
assert moved > 1e-6 and off < 1e-5, "stop: the LoRA switch does not work"
del head_t, arc_t, opt_t
for w in LORA:
    w.reset()
STEP_S = min(times[1:])
_steps = 6 * 5700 // 32 * 10 + 32 * 12900 // 32 * 10                 # R0, then every later training with XC's extra clips (an upper bound)
print(f"one training step: {STEP_S:.3f} s · about {_steps * STEP_S / 3600:.1f} h of training in all (upper bound), plus downloads and embeddings")

In [ ]:
# 10 · Scorers (LDN1, B1), notebook 11's splits with the fold rule, notebook 20's frozen references, and gate G0(b)
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


SCORE = {"B1": b1, "LDN1": ldn1}


def fold_of(p):
    return 1 if p in FOLDS[1] else 2


def fixed(E):
    return lambda p, s: E


def folded(E1, E2):
    # Split A: the model (or denoiser) of the machine's own fold; B and B+n: the other fold's, which never heard the machine
    return lambda p, s: (E1 if fold_of(p) == 1 else E2) if s == "A" else (E2 if fold_of(p) == 1 else E1)


def getter_of(E):
    return folded(*E) if isinstance(E, tuple) else fixed(E)


META, FZ = {}, {}
for t in TYPES:
    D = np.load(N20 / f"emb_{t}.npz", allow_pickle=False); Dn = np.load(N20 / f"emb_n2n_{t}.npz", allow_pickle=False)
    META[t] = (D["fname"], D["mid"], D["label"])
    for enc in ("BEATs", "ECHO"):
        FZ[(enc, t, "CH0")] = D[f"{enc}__CH0"].astype(np.float64)
        FZ[(enc, t, "MWF")] = D[f"{enc}__MWF"].astype(np.float64)
        FZ[(enc, t, "N2N-CH0")] = tuple(Dn[f"{enc}__N2N-CH0__{k}"].astype(np.float64) for k in (1, 2))
SPLIT = {t: {p: split_a(META[t][1], META[t][2], META[t][0], p) for p in sorted(set(META[t][1].tolist()))} for t in TYPES}
IS_TRAIN = {}
for t in TYPES:
    IS_TRAIN[t] = np.zeros(len(META[t][0]), bool)
    for p in SPLIT[t]:
        IS_TRAIN[t][SPLIT[t][p][0]] = True


def evaluate(get, t, scorer=ldn1):
    # notebook 11's splits; get(p, split) → the embedding matrix for machine p of type t
    ids = sorted(SPLIT[t]); label = META[t][2]
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = SPLIT[t][p]; y = label[te]
        fleet = np.concatenate([SPLIT[t][q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            E = get(p, s)
            r[s][p] = round(float(roc_auc_score(y, scorer(E[ref], E[te]))), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


def S_of(res, types):
    # the rules' score: mean over the types' machines of (A + B+10)/2 (each type has four machines)
    return round(float(np.mean([(res[t]["A"]["avg"] + res[t]["B+10"]["avg"]) / 2 for t in types])), 4)


N20RES = json.loads((N20 / "denoise_score.json").read_text())
FROZEN, worst = {}, 0.0
for tr_, inp in TRACK_INPUT.items():
    for sc in ("LDN1", "B1"):
        FROZEN[(tr_, sc)] = {t: evaluate(getter_of(FZ[("BEATs", t, inp)]), t, SCORE[sc]) for t in TYPES}
        for t in TYPES:
            ref = N20RES["results"][f"BEATs|{t}|{inp}|{sc}"]
            worst = max(worst, max(abs(FROZEN[(tr_, sc)][t][s][p] - ref[s][p]) for s in SPLITS for p in ref[s]))
FROZEN_S = {tr_: (S_of(FROZEN[(tr_, "LDN1")], MAIN), S_of(FROZEN[(tr_, "LDN1")], AUX)) for tr_ in TRACK_INPUT}
ref_gap = max(abs(FROZEN_S[tr_][i] - FROZEN_REF[tr_][i]) for tr_ in TRACK_INPUT for i in (0, 1))
G0 = {"a": {}, "b": {"worst_machine": round(worst, 4), "frozen_S": FROZEN_S, "gap_to_rules": round(ref_gap, 4)}, "c": {}}
print(f"G0(b) · notebook 20's frozen results recomputed: largest |ΔAUC| {worst:.4f} (tolerance {G0B_TOL})")
print("frozen references (development, confirmation):", FROZEN_S, f"· largest gap to the rules' numbers {ref_gap:.4f}")
if not TEST and (worst > G0B_TOL or ref_gap > G0B_TOL):
    raise SystemExit("stop: G0(b) failed, notebook 20's files do not give its numbers")

In [ ]:
# 11 · The audio: notebook 20's streams, its denoisers on microphone 0, the DCASE 2020 extra machines; gates G0(a), G0(c), GT
ST, XS, XMETA, XFZ, NETS, PDW, GT, XLEVEL, PREP_S, SAT = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
TRAIN_ROWS = {(t, p): SPLIT[t][p][0] for t in TYPES for p in SPLIT[t]}


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


_v = np.array([1.7, -2.3, 0.25], np.float32)
assert np.array_equal(f32(_v.astype(np.float16)), _v.astype(np.float16).astype(np.float32)) and abs(f32(np.int16([16384]))[0] - 0.5) < 1e-9


def mwf_weights(t):
    D = np.load(N19 / f"stats_{t}.npz", allow_pickle=False)
    return {k.split("__")[0]: D[k] for k in D.files if k.endswith("__w__MWF|-6dB")}


def build_streams(t, zfs, names, mid, W):
    # notebook 20's four streams, one pass over every clip: microphone 0 as int16 (exact), MWF as float16 (no clipping)
    n = len(names); ns = read2(zfs, names[0])["6dB"].shape[0]
    mm = {s: np.lib.format.open_memmap(STORE / f"{t}_{s}.npy", mode="w+", dtype=np.float16 if s in FLOAT_STREAMS else np.int16, shape=(n, ns))
          for s in STREAMS}
    Wt = {p: torch.from_numpy(w.astype(np.complex64)).to(DEVICE) for p, w in W.items()}
    peaks = {s: np.zeros(n, np.float32) for s in FLOAT_STREAMS}
    t0 = time.time()
    for i, x in enumerate(stream(lambda i: read2(zfs, names[i]), range(n))):
        p = mid[i]
        sig = {"CH0_m6": np.ascontiguousarray(x["-6dB"][:, 0]), "CH0_6": np.ascontiguousarray(x["6dB"][:, 0]),
               "MWF_m6": istft(beam(Wt[p], stft(to_dev(x["-6dB"]))), ns).cpu().numpy(),
               "MWF_6": istft(beam(Wt[p], stft(to_dev(x["6dB"]))), ns).cpu().numpy()}
        for s in STREAMS:
            if s in FLOAT_STREAMS:
                mm[s][i] = sig[s].astype(np.float16); peaks[s][i] = float(np.abs(sig[s]).max())
            else:
                mm[s][i] = to_int16(sig[s])
        if i % 1000 == 0:
            print(f"  {t}: {i}/{n} clips · {time.time() - t0:.0f} s", flush=True)
    for s in STREAMS:
        mm[s].flush()
    for s in FLOAT_STREAMS:
        SAT[f"{t}|{s}"] = peak_info(peaks[s], META[t][2])
    (STORE / f"{t}_peaks.json").write_text(json.dumps({s: SAT[f"{t}|{s}"] for s in FLOAT_STREAMS}))


def n2n_nets(t):
    # notebook 20's microphone-0 denoisers of this type, one per fold
    nets = {}
    for k in (1, 2):
        rec = torch.load(N20 / f"n2n_{t}_CH0_f{k}.pt", map_location="cpu", weights_only=False)
        assert rec["base"] == "CH0", rec["base"]
        net = N2NNet().to(DEVICE); net.load_state_dict(rec["state"]); net.eval(); nets[k] = net
    return nets


def peak_info(pk, labels=None):
    # how many clips reach |x| ≥ 1 (which 16-bit storage clips), overall and, for MIMII clips, by label
    out = {"clips_peak_at_least_1": round(float(np.mean(pk >= 1.0)), 4), "max_abs": round(float(pk.max()), 3)}
    if labels is not None:
        for nm, v in (("normal", 0), ("abnormal", 1)):
            out[nm] = round(float(np.mean(pk[labels == v] >= 1.0)), 4) if np.any(labels == v) else None
    return out


def apply_n2n(net, src_mm, path, batch=16, labels=None):
    # the denoiser on whole clips, written to a float16 stream (via a temporary file, so a crash leaves nothing half-written)
    tmp = path.with_name(path.stem + ".part.npy")
    n, ns = src_mm.shape
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, ns))
    pk = np.zeros(n, np.float32)
    for i in range(0, n, batch):
        y = denoise(net, torch.from_numpy(f32(src_mm[i:i + batch])).to(DEVICE)).cpu().numpy()
        mm[i:i + batch] = y.astype(np.float16); pk[i:i + len(y)] = np.abs(y).max(1)
    mm.flush(); del mm
    os.replace(tmp, path)
    info = peak_info(pk, labels)
    path.with_name(path.stem + "_peaks.json").write_text(json.dumps(info))
    return info


@torch.no_grad()
def frozen_emb(mm, rows=None, batch=None):
    # frozen BEATs (LoRA switched off) on whole clips of a stream → clip embeddings (n, 768) float32
    batch = batch or CFG["embed_batch"]; rows = np.arange(mm.shape[0]) if rows is None else rows
    LoRASwitch.on = False; model.eval(); out = []
    try:
        for i in range(0, len(rows), batch):
            x = torch.from_numpy(f32(mm[rows[i:i + batch]])).to(DEVICE)
            out.append(tokens(x, amp=AMP).mean(1).cpu().numpy())
    finally:
        LoRASwitch.on = True
    return np.concatenate(out)


def build_xc(t):
    # DCASE 2020's extra machines of this type: the first CFG["xc_cap"] normal clips by name per machine, microphone 0 at −6 dB
    meta_f = STORE / f"xc_{t}.json"
    if meta_f.exists() and all((STORE / f"xc_{t}_{s}.npy").exists() for s in ("CH0", "N2N1", "N2N2")):
        return json.loads(meta_f.read_text())
    zf = zipfile.ZipFile(get_file(XZIPS[t]))
    names = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
    mid_all = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    if not TEST:
        assert all(n.replace("\\", "/").split("/")[-1].startswith("normal") for n in names), "stop: the additional training data should be normal clips only"
    keep = np.concatenate([np.where(mid_all == q)[0][:CFG["xc_cap"]] for q in sorted(set(mid_all.tolist()))])
    names, mid = names[keep], mid_all[keep]
    assert not (set(mid.tolist()) & set(META[t][1].tolist())), "stop: an extra machine has a MIMII machine's ID"
    ns = read_all(zf.read(names[0]))[0].shape[0]
    mm = np.lib.format.open_memmap(STORE / f"xc_{t}_CH0.npy", mode="w+", dtype=np.int16, shape=(len(names), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zf.read(names[i])), range(len(names)))):
        assert sr == SR and x.shape[0] == ns, (names[i], sr, x.shape)
        mm[i] = to_int16(x[:, 0])
    mm.flush(); del mm; zf.close()
    src_mm = np.load(STORE / f"xc_{t}_CH0.npy", mmap_mode="r")
    for k in (1, 2):
        SAT[f"{t}|XC_N2N{k}"] = apply_n2n(NETS[t][k], src_mm, STORE / f"xc_{t}_N2N{k}.npy")
    info = {"names": names.tolist(), "mid": mid.tolist(), "per_machine": dict(Counter(mid.tolist())), "samples": int(ns),
            "zip": XZIPS[t][1], "zip_md5": XZIPS[t][2]}
    meta_f.write_text(json.dumps(info))
    if IN_COLAB:
        (WORK / XZIPS[t][1]).unlink(missing_ok=True)
    return info


def twin_power(t, dom):
    # the mean power of the noise-only twin difference y₋₆ − y₆, for training clips only (a test clip's twin is never read)
    a, b = ST[t][f"{dom}_m6"], ST[t][f"{dom}_6"]
    P = np.full(a.shape[0], np.nan)
    rows = np.where(IS_TRAIN[t])[0]
    for i in range(0, len(rows), 64):
        r = rows[i:i + 64]
        P[r] = np.mean((f32(a[r]).astype(np.float64) - f32(b[r])) ** 2, 1)
    return P


def swap_full(t, dom, i, j, shift):
    # TS on whole clips: clip i's 6 dB twin plus the power-matched, circularly shifted noise of clip j (both training clips)
    assert IS_TRAIN[t][i] and IS_TRAIN[t][j] and META[t][1][i] == META[t][1][j]
    st = ST[t]; g = math.sqrt(PDW[(t, dom)][i] / PDW[(t, dom)][j])
    D = f32(st[f"{dom}_m6"][j]) - f32(st[f"{dom}_6"][j])
    return f32(st[f"{dom}_6"][i]) + C_SWAP * g * np.roll(D, shift)


def gate_gt(t):
    rng = np.random.default_rng(0)
    for dom in ("MWF", "CH0"):
        ratios = []
        for _ in range(GT_PAIRS):
            p = sorted(SPLIT[t])[int(rng.integers(0, len(SPLIT[t])))]
            i, j = rng.choice(TRAIN_ROWS[(t, p)], 2, replace=False)
            y = swap_full(t, dom, int(i), int(j), int(rng.integers(0, ST[t][f"{dom}_m6"].shape[1])))
            ratios.append(float(np.mean(y.astype(np.float64) ** 2) / np.mean(f32(ST[t][f"{dom}_m6"][i]).astype(np.float64) ** 2)))
        GT[(t, dom)] = round(float(np.median(ratios)), 4)
    print(f"GT · {t}: median power of the swap over the −6 dB clip: MWF {GT[(t, 'MWF')]}, microphone 0 {GT[(t, 'CH0')]} (range {GT_RANGE})")


def gate_g0c(t):
    m = min(G0C_CLIPS, len(META[t][0])); rows = np.arange(m)
    pairs = {"CH0": (ST[t]["CH0_m6"], FZ[("BEATs", t, "CH0")]), "MWF": (ST[t]["MWF_m6"], FZ[("BEATs", t, "MWF")]),
             "N2N-CH0 fold 1": (ST[t]["N2N1_m6"], FZ[("BEATs", t, "N2N-CH0")][0]),
             "N2N-CH0 fold 2": (ST[t]["N2N2_m6"], FZ[("BEATs", t, "N2N-CH0")][1])}
    t0, cos, worst = time.time(), {}, {}
    for k, (mm, ref) in pairs.items():
        e = frozen_emb(mm, rows)
        c = np.sum(unit(e.astype(np.float64)) * unit(ref[:m]), 1); w = int(np.argmin(c))
        cos[k] = round(float(c[w]), 6)
        worst[k] = {"clip": str(META[t][0][w]), "cosine": round(float(c[w]), 6), "peak_abs": round(float(np.abs(f32(mm[w])).max()), 3)}
    G0["c"][t] = cos; G0.setdefault("c_worst", {})[t] = worst
    PREP_S["embed_clips_per_s"] = round(4 * m / (time.time() - t0), 1)
    print(f"G0(c) · {t}: frozen BEATs on our streams against notebook 20's embeddings, smallest cosine {cos} (bar {G0C_COS})")
    if not TEST and min(cos.values()) < G0C_COS:
        print("the worst clip of each stream:", json.dumps(worst, indent=1))
        raise SystemExit("stop: G0(c) failed, the streams or the denoisers are not notebook 20's")


def prepare(t, nxt=None):
    t0 = time.time(); names, mid, label = META[t]
    NETS[t] = n2n_nets(t)
    if not (STORE / f"{t}.json").exists():
        W = mwf_weights(t)
        assert set(W) == set(mid.tolist()), f"{t}: MWF weights for {sorted(W)} but machines {sorted(set(mid.tolist()))}"
        if t in PREFETCH:
            PREFETCH[t].join()
        zfs = open2(t)
        ok = all(np.array_equal(np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav"))), names) for zf in zfs.values())
        G0["a"][t] = bool(ok)
        print(f"G0(a) · {t}: the clips are notebook 20's: {ok}")
        if not ok and not TEST:
            raise SystemExit("stop: G0(a) failed")
        build_streams(t, zfs, names, mid, W)
        for zf in zfs.values():
            zf.close()
        if IN_COLAB:
            for lv in LEVELS:
                (WORK / ZIPS[f"{t}|{lv}"][1]).unlink(missing_ok=True)
        (STORE / f"{t}.json").write_text(json.dumps({"n": int(len(names))}))
    else:
        G0["a"].setdefault(t, "local store from an earlier run of this notebook in this session")
        SAT.update({f"{t}|{s}": v for s, v in json.loads((STORE / f"{t}_peaks.json").read_text()).items()})
    prefetch(nxt)
    ST[t] = {s: np.load(STORE / f"{t}_{s}.npy", mmap_mode="r") for s in STREAMS}
    for k in (1, 2):
        pth = STORE / f"{t}_N2N{k}_m6.npy"
        if not pth.exists():
            apply_n2n(NETS[t][k], ST[t]["CH0_m6"], pth, labels=label)
        SAT[f"{t}|N2N{k}_m6"] = json.loads(pth.with_name(pth.stem + "_peaks.json").read_text())
        ST[t][f"N2N{k}_m6"] = np.load(pth, mmap_mode="r")
    XMETA[t] = build_xc(t)
    XS[t] = {s: np.load(STORE / f"xc_{t}_{s}.npy", mmap_mode="r") for s in ("CH0", "N2N1", "N2N2")}
    gate_g0c(t)
    for dom in ("MWF", "CH0"):
        PDW[(t, dom)] = twin_power(t, dom)
    gate_gt(t)
    rms = lambda mm, rows: float(np.median(np.sqrt(np.mean(f32(mm[rows]).astype(np.float64) ** 2, 1))))
    tr_rows = np.where(IS_TRAIN[t])[0][:200]
    XLEVEL[t] = round(rms(XS[t]["CH0"], np.arange(min(200, len(XMETA[t]["mid"])))) / rms(ST[t]["CH0_m6"], tr_rows), 3)
    xf = OUT / f"xc_frozen2_{t}.npz"
    if not xf.exists():
        np.savez(xf, **{s: frozen_emb(XS[t][s]).astype(np.float16) for s in ("CH0", "N2N1", "N2N2")})
    Dx = np.load(xf, allow_pickle=False); XFZ[t] = {s: Dx[s].astype(np.float64) for s in Dx.files}
    PREP_S[t] = round(time.time() - t0, 1)
    print(f"{t}: ready · {len(names)} MIMII clips · extra machines {XMETA[t]['per_machine']} (RMS ratio to MIMII −6 dB {XLEVEL[t]})"
          f" · {PREP_S[t]:.0f} s", flush=True)
    print("  share of clips whose processed signal peaks at |x| ≥ 1 (16-bit storage would have clipped them):",
          {k.split("|")[1]: v["clips_peak_at_least_1"] for k, v in SAT.items() if k.startswith(t + "|")})


for j_, t in enumerate(TYPES):
    prepare(t, nxt=TYPES[j_ + 1] if j_ + 1 < len(TYPES) else None)
GT_OK = {tr: all(GT_RANGE[0] <= GT[(t, TWIN_DOMAIN[tr])] <= GT_RANGE[1] for t in TYPES) for tr in TWIN_DOMAIN}
print("GT · TS can be tested on:", GT_OK, f"· all data ready after {(time.time() - T_START) / 60:.0f} min")
_n_train = 2 * sum(1 + sum(1 for g in ("XC", "PL", "TS", "PD") if tr in APPLIES[g]) + 2 for tr in TRACKS)
_clips = sum(len(META[t][0]) for t in TYPES)
_xc = sum(len(XMETA[t]["mid"]) for t in TYPES)
_steps = (sum(len(TRAIN_ROWS[k]) for k in TRAIN_ROWS) / 2 + _xc) / CFG["batch"] * CFG["epochs"]
_h = _n_train * (_steps * STEP_S * 1.15 + _clips / PREP_S["embed_clips_per_s"]) / 3600
print(f"estimate for the program: {_n_train} trainings · about {_h:.1f} h on this GPU (upper bound: every training with XC's extra clips)")

In [ ]:
# 12 · Training (one function for every recipe), embeddings, and the scores of a system (recipe, seeds, fusion)
TRK = {"ARR": "ARR", "MIC+": "MICp", "MIC": "MIC"}                   # file-name tags


def tag_of(track, flags, seed, fold):
    return f"{TRK[track]}_{'-'.join(f for f in ('XC', 'PL', 'TS', 'PD') if f in flags) or 'R0'}_s{seed}_f{fold}"


def frozen_rows_emb(track, fold, t, xc):
    # the frozen BEATs embeddings PL clusters on: the track's input (MIC+: the fold's denoiser); XC machines: this notebook's
    if xc:
        return XFZ[t]["CH0" if track == "MIC" else f"N2N{fold}"]
    E = FZ[("BEATs", t, TRACK_INPUT[track])]
    return E[fold - 1] if isinstance(E, tuple) else E


def training_set(track, flags, fold):
    # items: (type index, source 0 = MIMII / 1 = XC, row, class); one class per machine, or per k-means sub-class with PL
    items, classes, pl_sizes = [], [], {}

    def add(ti, source, rows, name, E):
        if "PL" in flags:
            lab = KMeans(n_clusters=CFG["pl_k"], n_init=CFG["pl_starts"], random_state=CFG["pl_seed"]).fit(unit(E[rows])).labels_
            base = len(classes); classes.extend(f"{name}|k{c}" for c in range(CFG["pl_k"]))
            items.extend((ti, source, int(r), base + int(c)) for r, c in zip(rows, lab))
            pl_sizes[name] = sorted(Counter(lab.tolist()).values())
        else:
            c = len(classes); classes.append(name); items.extend((ti, source, int(r), c) for r in rows)
    for ti, t in enumerate(TYPES):
        for p in FOLDS[fold]:
            if p in SPLIT[t]:
                add(ti, 0, TRAIN_ROWS[(t, p)], f"{t}|{p}", frozen_rows_emb(track, fold, t, False))
    if "XC" in flags:
        for ti, t in enumerate(TYPES):
            xm = np.array(XMETA[t]["mid"])
            for q in sorted(set(xm.tolist())):
                add(ti, 1, np.where(xm == q)[0], f"{t}|{q}|DCASE2020", frozen_rows_emb(track, fold, t, True))
    return np.array(items, dtype=np.int64), classes, pl_sizes


def win(mm, row, s, n):
    return f32(mm[row, s:s + n])


def rolled_win(t, dom, j, s, n, shift):
    # window [s, s + n) of clip j's noise-only difference after a circular shift
    st = ST[t]; N = st[f"{dom}_m6"].shape[1]; idx = (np.arange(s, s + n) - shift) % N
    return f32(st[f"{dom}_m6"][j][idx]) - f32(st[f"{dom}_6"][j][idx])


def make_batch(track, flags, fold, items, ids, rng, n, want_teacher):
    # the student's input crops, PD's teacher crops (the 6 dB twin of the same window) and which rows have a teacher
    B = len(ids); x = np.zeros((B, n), np.float32); tch = np.zeros((B, n), np.float32) if want_teacher else None
    has = np.zeros(B, bool); full = {}; n_swap = 0
    for b, it in enumerate(ids):
        ti, source, r, _ = items[it]; t = TYPES[ti]
        if source == 1:
            mm = XS[t]["CH0" if track == "MIC" else f"N2N{fold}"]
            x[b] = win(mm, r, int(rng.integers(0, mm.shape[1] - n + 1)), n)
            continue
        st = ST[t]; N = st["CH0_m6"].shape[1]; s = int(rng.integers(0, N - n + 1))
        swap = "TS" in flags and rng.random() < CFG["ts_prob"]
        if swap:
            pool = TRAIN_ROWS[(t, META[t][1][r])]; j = int(rng.choice(pool[pool != r])); shift = int(rng.integers(0, N)); n_swap += 1
        if track == "ARR":
            if swap:
                assert IS_TRAIN[t][r] and IS_TRAIN[t][j]
                g = math.sqrt(PDW[(t, "MWF")][r] / PDW[(t, "MWF")][j])
                x[b] = win(st["MWF_6"], r, s, n) + C_SWAP * g * rolled_win(t, "MWF", j, s, n, shift)
            else:
                x[b] = win(st["MWF_m6"], r, s, n)
            if want_teacher:
                assert IS_TRAIN[t][r]
                tch[b] = win(st["MWF_6"], r, s, n); has[b] = True
        elif track == "MIC+":
            if swap:
                full.setdefault(t, []).append((b, s, swap_full(t, "CH0", r, j, shift)))
            else:
                x[b] = win(st[f"N2N{fold}_m6"], r, s, n)
            if want_teacher:
                assert IS_TRAIN[t][r]
                tch[b] = win(st["CH0_6"], r, s, n); has[b] = True
        else:
            x[b] = win(st["CH0_m6"], r, s, n)
    X = torch.from_numpy(x).to(DEVICE)
    for t, lst in full.items():                                       # MIC+ swaps: the fold's denoiser on the whole mixture, then the crop
        Y = denoise(NETS[t][fold], torch.from_numpy(np.stack([z for _, _, z in lst])).to(DEVICE))
        for (b, s, _), y in zip(lst, Y):
            X[b] = y[s:s + n]
    T = torch.from_numpy(tch).to(DEVICE) if want_teacher else None
    return X, T, torch.from_numpy(has).to(DEVICE), n_swap


def adapter_state(head, arc):
    return {"lora": [{"A": w.A.detach().cpu(), "B": w.B.detach().cpu()} for w in LORA],
            "head": {k: v.cpu() for k, v in head.state_dict().items()}, "arc": {k: v.cpu() for k, v in arc.state_dict().items()}}


def load_adapters(st, head, arc):
    with torch.no_grad():
        for w, s in zip(LORA, st["lora"]):
            w.A.copy_(s["A"].to(DEVICE)); w.B.copy_(s["B"].to(DEVICE))
    head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])


def teacher_tokens(T):
    # frozen BEATs (LoRA off, evaluation mode, no gradient) on the teacher crops
    with torch.no_grad():
        LoRASwitch.on = False; model.eval()
        try:
            return tokens(T, amp=AMP)
        finally:
            LoRASwitch.on = True; model.train()


def step_losses(arc, head, X, y, mix, lam, perm, T, has, lam_pd):
    tok = tokens(X, amp=AMP)
    cos = arc.cosine(head(tok.mean(1)))
    ce = lam * arc.loss(cos, y) + (1.0 - lam) * arc.loss(cos, y[perm]) if mix else arc.loss(cos, y)
    pd = None
    if T is not None and bool(has.any()):
        tt = teacher_tokens(T[has])
        pd = (1.0 - F.cosine_similarity(tok[has], tt, dim=-1)).mean()
    return ce, pd


def train(track, flags, seed, fold):
    tag = tag_of(track, flags, seed, fold)
    final, resume = OUT / f"ft_{tag}.pt", OUT / f"ft_{tag}_resume.pt"
    items, classes, pl_sizes = training_set(track, flags, fold)
    head, arc = fresh_adapters(seed, len(classes))
    if final.exists():
        st = torch.load(final, map_location="cpu", weights_only=False)
        assert st["classes"] == classes, "the saved model used other classes"
        load_adapters(st, head, arc); print(final.name, "already in Drive: loaded, not trained again")
        return head, st["log"]
    pd_on = "PD" in flags
    params = lora_params() + list(head.parameters()) + list(arc.parameters())
    opt = torch.optim.AdamW([{"params": lora_params(), "lr": CFG["lr_lora"], "weight_decay": 0.0},
                             {"params": list(head.parameters()) + list(arc.parameters()), "lr": CFG["lr_head"], "weight_decay": CFG["wd_head"]}])
    spe = len(items) // CFG["batch"]
    total, warm = spe * CFG["epochs"], spe * CFG["warmup_epochs"]
    lr_f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1.0 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, [lr_f, lr_f])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"tag": tag, "track": track, "flags": list(flags), "seed": seed, "fold": fold, "cfg": CFG, "classes": classes,
           "pl_sizes": pl_sizes, "n_train": int(len(items)), "n_xc": int((items[:, 1] == 1).sum()), "lambda_pd": None,
           "epochs": [], "resumed_at_epoch": []}
    start, n = 0, CROP
    if resume.exists():
        st = torch.load(resume, map_location="cpu", weights_only=False)
        load_adapters(st, head, arc)
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        log, start = st["log"], st["epoch"] + 1
        log["resumed_at_epoch"].append(start); print(f"resuming {tag} at epoch {start}")
    if pd_on and log["lambda_pd"] is None:
        # the rules' balance: λ_PD = mean ArcFace loss / mean distillation loss over a few batches, before any update
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), 999]); ce_l, pd_l = [], []
        model.train(); head.train(); arc.train()
        with torch.no_grad():
            for _ in range(CFG["pd_balance_batches"]):
                ids = rng.choice(len(items), CFG["batch"], replace=False)
                X, T, has, _ = make_batch(track, flags, fold, items, ids, rng, n, True)
                y = torch.from_numpy(items[ids, 3]).to(DEVICE)
                ce, pd = step_losses(arc, head, X, y, False, 1.0, None, T, has, 0.0)
                ce_l.append(float(ce)); pd_l.append(float(pd)) if pd is not None else None
        log["lambda_pd"] = round(float(np.mean(ce_l) / max(np.mean(pd_l), 1e-6)), 4)
        print(f"{tag}: λ_PD = {log['lambda_pd']} (ArcFace {np.mean(ce_l):.3f}, distillation {np.mean(pd_l):.4f} at the start)")
    print(f"{tag}: {len(items)} training clips ({log['n_xc']} from DCASE 2020) · {len(classes)} classes · {spe} steps per epoch", flush=True)
    for ep in range(start, CFG["epochs"]):
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), ep])
        np.random.seed(seed * 1000 + ep)                              # BEATs draws its layer-drop from numpy's global stream
        order = rng.permutation(len(items))
        model.train(); head.train(); arc.train()
        t0, tot, tce, tpd, n_mix, n_sw = time.time(), 0.0, 0.0, 0.0, 0, 0
        for b in range(spe):
            ids = order[b * CFG["batch"]:(b + 1) * CFG["batch"]]
            mix = bool(rng.random() < CFG["mixup_prob"])
            X, T, has, ns_ = make_batch(track, flags, fold, items, ids, rng, n, pd_on)
            y = torch.from_numpy(items[ids, 3]).to(DEVICE)
            lam, perm = 1.0, None
            if mix:
                lam = float(rng.beta(CFG["mixup_beta"], CFG["mixup_beta"]))
                perm = torch.from_numpy(rng.permutation(len(ids))).to(DEVICE)
                X = lam * X + (1.0 - lam) * X[perm]
                if pd_on:
                    T = lam * T + (1.0 - lam) * T[perm]; has = has & has[perm]
            ce, pd = step_losses(arc, head, X, y, mix, lam, perm, T, has, log["lambda_pd"])
            loss = ce if pd is None else ce + log["lambda_pd"] * pd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, CFG["clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); tce += float(ce.detach()); tpd += 0.0 if pd is None else float(pd.detach())
            n_mix += mix; n_sw += ns_
        log["epochs"].append({"epoch": ep, "mean_loss": round(tot / spe, 4), "mean_arcface": round(tce / spe, 4),
                              "mean_distillation": round(tpd / spe, 5), "mixup_batches": int(n_mix), "swaps": int(n_sw),
                              "seconds": round(time.time() - t0, 1)})
        print(f"{tag} · epoch {ep + 1}/{CFG['epochs']} · loss {tot / spe:.3f}" + (f" (distillation {tpd / spe:.4f})" if pd_on else "")
              + f" · {time.time() - t0:.0f} s", flush=True)
        torch.save({**adapter_state(head, arc), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "log": log}, resume)
    model.eval(); head.eval(); arc.eval()
    torch.save({**adapter_state(head, arc), "classes": classes, "log": log}, final)
    resume.unlink(missing_ok=True)
    return head, log


def input_stream(track, fold, t):
    return ST[t]["MWF_m6" if track == "ARR" else (f"N2N{fold}_m6" if track == "MIC+" else "CH0_m6")]


@torch.no_grad()
def embed(track, fold, head, keep_pooled):
    # whole 10-second clips of every MIMII clip through the trained model: F (256, the head) and P (768, the pooled backbone)
    model.eval(); head.eval(); LoRASwitch.on = True; out = {}
    for t in TYPES:
        mm = input_stream(track, fold, t); nn_ = mm.shape[0]
        Fm = np.zeros((nn_, CFG["emb_dim"]), np.float16); Pm = np.zeros((nn_, 768), np.float16)
        for i in range(0, nn_, CFG["embed_batch"]):
            x = torch.from_numpy(f32(mm[i:i + CFG["embed_batch"]])).to(DEVICE)
            pooled = tokens(x, amp=AMP).mean(1)
            Fm[i:i + len(x)] = head(pooled).cpu().numpy(); Pm[i:i + len(x)] = pooled.cpu().numpy()
        out[f"F__{t}"] = Fm
        if keep_pooled:
            out[f"P__{t}"] = Pm
    return out


LOGS, T_TRAIN = {}, {}


def ensure(track, flags, seed):
    # both folds of one recipe and seed: trained (or resumed, or read from Drive) and embedded; returns gate G1 for both
    ok = True
    for fold in (1, 2):
        tag = tag_of(track, flags, seed, fold); ef, fin = OUT / f"emb_{tag}.npz", OUT / f"ft_{tag}.pt"
        if ef.exists() and fin.exists():
            log = torch.load(fin, map_location="cpu", weights_only=False)["log"]
        else:
            t0 = time.time()
            head, log = train(track, flags, seed, fold)
            E = embed(track, fold, head, keep_pooled=True)
            tmp = OUT / f"emb_{tag}.part.npz"
            np.savez(tmp, **E, meta=json.dumps({"tag": tag, "weights_md5": md5(fin), "checkpoint_sha256": CKPT_SHA, "test_run": TEST,
                                                "date": time.strftime("%Y-%m-%d %H:%M")}))
            os.replace(tmp, ef)
            T_TRAIN[tag] = round(time.time() - t0, 1)
            print(f"saved {ef.name} · {T_TRAIN[tag] / 60:.1f} min for training and embedding", flush=True)
            del head; gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        LOGS[tag] = log
        ok = ok and log["epochs"][-1]["mean_loss"] < log["epochs"][0]["mean_loss"]
    return ok


EMB = {}


def emb_of(track, flags, seed, kind="F"):
    key = (track, tuple(flags), seed, kind)
    if key not in EMB:
        Es = []
        for fold in (1, 2):
            D = np.load(OUT / f"emb_{tag_of(track, flags, seed, fold)}.npz", allow_pickle=False)
            Es.append({t: D[f"{kind}__{t}"].astype(np.float32) for t in TYPES})
        EMB[key] = Es
    return EMB[key]


def system(track, flags, se=False, ef=False, kind="F"):
    # per type, the getter of the system: seeds averaged (SE), ECHO fused (EF); model k always pairs with denoiser k
    seeds = CFG["se_seeds"] if se else [0]
    per_seed = [emb_of(track, flags, s, kind) for s in seeds]
    G = {}
    for t in TYPES:
        Ek = []
        for k in (1, 2):
            E = np.concatenate([unit(ps[k - 1][t]) for ps in per_seed], 1) / math.sqrt(len(per_seed))
            if ef:
                e = FZ[("ECHO", t, TRACK_INPUT[track])]; e = e[k - 1] if isinstance(e, tuple) else e
                E = np.concatenate([E, unit(e)], 1) / math.sqrt(2.0)
            Ek.append(E)
        G[t] = folded(Ek[0], Ek[1])
    return G


RES = {}


def res_of(track, flags, se=False, ef=False, scorer="LDN1", kind="F"):
    key = (track, tuple(flags), bool(se), bool(ef), scorer, kind)
    if key not in RES:
        G = system(track, flags, se, ef, kind)
        RES[key] = {t: evaluate(G[t], t, SCORE[scorer]) for t in TYPES}
    return RES[key]


def dev_conf(res):
    return S_of(res, MAIN), S_of(res, AUX)


def res_sys(track, c, scorer="LDN1"):
    # the scores of a system c = {flags, se, ef, kind}
    return res_of(track, c["flags"], c["se"], c["ef"], scorer, c["kind"])

In [ ]:
# 13 · The program, exactly as in the rules: R0, then EP → XC → PL → TS → PD → SE → EF (each kept only if it passes), then FF
CUR = {tr: {"flags": [], "se": False, "ef": False, "kind": "F"} for tr in TRACKS}
STEPS = {tr: [] for tr in TRACKS}
R0_OK = {}
t_prog = time.time()
for tr in TRACKS:
    R0_OK[tr] = ensure(tr, [], 0)
    d, c = dev_conf(res_sys(tr, CUR[tr]))
    STEPS[tr].append({"step": "R0", "decidable": bool(R0_OK[tr]), "dev": d, "conf": c, "kept": True})
    print(f"{tr} · R0: development {d} · confirmation {c} · frozen BEATs {FROZEN_S[tr]} · G1 {R0_OK[tr]}", flush=True)

for ing in ORDER:
    for tr in TRACKS:
        if tr not in APPLIES[ing]:
            continue
        cur = dict(CUR[tr]); ref_d, ref_c = dev_conf(res_sys(tr, cur))
        rec = {"step": ing, "current": {k: (list(v) if isinstance(v, list) else v) for k, v in cur.items()}, "ref_dev": ref_d, "ref_conf": ref_c}
        if not R0_OK[tr]:
            rec.update(decidable=False, kept=False, why="R0 failed G1 on this track"); STEPS[tr].append(rec); continue
        if ing == "TS" and not GT_OK[tr]:
            rec.update(decidable=False, kept=False, why="GT failed: the swap does not keep the −6 dB power"); STEPS[tr].append(rec); continue
        if ing == "EP":
            new = {**cur, "kind": "P"}; ok = True
        elif ing in ("XC", "PL", "TS", "PD"):
            new = {**cur, "flags": cur["flags"] + [ing]}
            ok = ensure(tr, new["flags"], 0)
        elif ing == "SE":
            new = {**cur, "se": True}
            ok = ensure(tr, cur["flags"], CFG["se_seeds"][1]) and ensure(tr, cur["flags"], CFG["se_seeds"][2])
        else:
            new = {**cur, "ef": True}; ok = True
        d, c = dev_conf(res_sys(tr, new))
        kept = bool(ok and d >= round(ref_d + BAR, 4) and c >= round(ref_c - BAR, 4))
        rec.update(decidable=bool(ok), dev=d, conf=c, bar=round(ref_d + BAR, 4), conf_bar=round(ref_c - BAR, 4), kept=kept)
        if kept:
            CUR[tr] = new
        STEPS[tr].append(rec)
        print(f"{tr} · {ing}: development {d} against {ref_d} + {BAR} · confirmation {c} against {ref_c} − {BAR}"
              f" · {'kept' if kept else ('not decidable (G1)' if not ok else 'not kept')}", flush=True)


def describe(c):
    return ("+".join(c["flags"]) or "R0") + (" +SE" if c["se"] else "") + (" +EF" if c["ef"] else "") + f" on {c['kind']}"


FF, FINAL = {}, {}
for tr in TRACKS:
    cur = CUR[tr]; d, c = dev_conf(res_sys(tr, cur)); fd, fc = FROZEN_S[tr]
    passed = bool(R0_OK[tr] and d >= round(fd + BAR, 4) and c >= round(fc - BAR, 4))
    FF[tr] = {"system": dict(cur), "describe": describe(cur), "dev": d, "conf": c, "frozen_dev": fd, "frozen_conf": fc,
              "bar": round(fd + BAR, 4), "conf_bar": round(fc - BAR, 4), "pass": passed}
    FINAL[tr] = ("fine-tuned", cur) if passed else ("frozen BEATs LDN1", None)
    print(f"FF · {tr}: final {describe(cur)}: development {d} against frozen {fd} + {BAR} · confirmation {c} against {fc} − {BAR} →",
          "fine-tuned from here on" if passed else "frozen BEATs stays")
print(f"program done · {(time.time() - t_prog) / 60:.0f} min in this session")

In [ ]:
# 14 · Reported numbers, save, two pictures, and one zip to download
import matplotlib
import matplotlib.pyplot as plt


def final_res(tr, scorer="LDN1"):
    kind_, cur = FINAL[tr]
    if cur is None:
        return FROZEN[(tr, scorer)]
    return res_sys(tr, cur, scorer)


REP = {"per_track": {}, "T1_vs_published": {}, "T2_arr_b10": {}, "seed_spread": {}, "frozen_beats_echo_LDN1": {}, "n16_context": N16_CONTEXT}
for tr in TRACKS:
    cur = CUR[tr]
    recipes = [[]] + [r["current"]["flags"] + [r["step"]] for r in STEPS[tr] if r["step"] in ("XC", "PL", "TS", "PD")]
    rows = {}
    for fl in recipes:                                                # every seed-0 recipe that was trained: LDN1 and B1, on F and on P
        if not all((OUT / f"emb_{tag_of(tr, fl, 0, k)}.npz").exists() for k in (1, 2)):
            continue
        name = "+".join(fl) or "R0"
        for sc, kind in (("LDN1", "F"), ("B1", "F"), ("LDN1", "P"), ("B1", "P")):
            r = res_of(tr, fl, False, False, sc, kind)
            rows[f"{name}|{sc}|{kind}"] = {**{t: {s: r[t][s]["avg"] for s in ("A", "B+10")} for t in TYPES}, "dev_conf": dev_conf(r)}
    REP["per_track"][tr] = rows
    fr = final_res(tr)
    REP["T1_vs_published"][tr] = {t: {"published": PUBLISHED_M6[t], "final": fr[t]["A"]["avg"], "frozen": FROZEN[(tr, "LDN1")][t]["A"]["avg"],
                                      "like_for_like": tr == "MIC"} for t in TYPES}
    if all((OUT / f"emb_{tag_of(tr, cur['flags'], s, k)}.npz").exists() for s in CFG["se_seeds"] for k in (1, 2)):
        REP["seed_spread"][tr] = {s: dev_conf({t: evaluate(folded(*[emb_of(tr, cur["flags"], s, cur["kind"])[k][t] for k in (0, 1)]), t)
                                               for t in TYPES}) for s in CFG["se_seeds"]}
    G = {}
    for t in TYPES:
        b, e = FZ[("BEATs", t, TRACK_INPUT[tr])], FZ[("ECHO", t, TRACK_INPUT[tr])]
        if isinstance(b, tuple):
            G[t] = folded(*[np.concatenate([unit(b[k]), unit(e[k])], 1) / math.sqrt(2.0) for k in (0, 1)])
        else:
            G[t] = fixed(np.concatenate([unit(b), unit(e)], 1) / math.sqrt(2.0))
    REP["frozen_beats_echo_LDN1"][tr] = dev_conf({t: evaluate(G[t], t) for t in TYPES})
if "ARR" in TRACKS:
    fa = final_res("ARR")
    REP["T2_arr_b10"] = {t: {"target_6dB": T2_TARGET[t], "final": fa[t]["B+10"]["avg"], "frozen": FROZEN[("ARR", "LDN1")][t]["B+10"]["avg"]} for t in MAIN}

print(f"{'track':5s} {'system':22s} {'type':7s} " + " ".join(f"{s:>7s}" for s in SPLITS))
for tr in TRACKS:
    for nm, r in (("frozen BEATs LDN1", FROZEN[(tr, "LDN1")]), ("final " + FINAL[tr][0], final_res(tr))):
        for t in TYPES:
            print(f"{tr:5s} {nm[:22]:22s} {t:7s} " + " ".join(f"{r[t][s]['avg']:7.4f}" for s in SPLITS))
print("\nT1 · Split A at −6 dB against arXiv 2309.07500 (like for like only on MIC):")
for tr in TRACKS:
    print(f"  {tr:5s}", {t: (v["final"], v["published"]) for t, v in REP["T1_vs_published"][tr].items()})
print("T2 · ARR B+10 against the 6 dB targets:", REP["T2_arr_b10"])
print("seed spread (development, confirmation):", REP["seed_spread"])
print("frozen BEATs + ECHO (LDN1):", REP["frozen_beats_echo_LDN1"])

RESULT = {"rules": RULES, "program": PROGRAM, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "torch": torch.__version__, "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
                   "zips": {k: v[1:] for k, v in ZIPS.items()}, "xc_zips": {k: v[1:] for k, v in XZIPS.items()}, "cfg": CFG,
                   "c_swap": C_SWAP, "step_seconds": STEP_S, "prep_seconds": PREP_S, "train_seconds": T_TRAIN, "drive_free_gb": round(DRIVE_FREE_GB, 1)},
          "G0": G0, "GT": {f"{t}|{d}": v for (t, d), v in GT.items()}, "GT_ok": GT_OK, "peaks": SAT,
          "G1": {tag: [lg["epochs"][0]["mean_loss"], lg["epochs"][-1]["mean_loss"]] for tag, lg in LOGS.items()},
          "lambda_pd": {tag: lg["lambda_pd"] for tag, lg in LOGS.items() if lg.get("lambda_pd") is not None},
          "xc": {t: {"per_machine": XMETA[t]["per_machine"], "rms_ratio_to_mimii_m6": XLEVEL[t]} for t in TYPES},
          "frozen": {tr: {"S": FROZEN_S[tr], **{sc: FROZEN[(tr, sc)] for sc in ("LDN1", "B1")}} for tr in TRACKS},
          "steps": STEPS, "FF": FF, "final": {tr: {"kind": FINAL[tr][0], "system": FINAL[tr][1],
                                                  "describe": describe(FINAL[tr][1]) if FINAL[tr][1] else "frozen BEATs LDN1"} for tr in TRACKS},
          "reported": REP, "results": {"|".join(map(str, k[:1])) + "|" + ("+".join(k[1]) or "R0") + f"|se={k[2]}|ef={k[3]}|{k[4]}|{k[5]}": v
                                       for k, v in RES.items()},
          "training_logs": LOGS}
(OUT / "strong_finetune.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
fig, axes = plt.subplots(1, len(TRACKS), figsize=(5.2 * len(TRACKS), 4.2), sharey=False)
axes = np.atleast_1d(axes)
for ax, tr in zip(axes, TRACKS):
    st = [r for r in STEPS[tr] if r.get("dev") is not None]
    xs = np.arange(len(st)); labels = [r["step"] for r in st]
    ax.plot(xs, [r["dev"] for r in st], "-o", color="#2a78d6", lw=2, ms=6, label="development (pumps, fans)")
    ax.plot(xs, [r["conf"] for r in st], "--s", color="#eb6834", lw=1.5, ms=5, label="confirmation (valves, slide rails)")
    for x_, r in zip(xs, st):
        if r["step"] != "R0" and not r["kept"]:
            ax.plot([x_], [r["dev"]], "o", ms=9, mfc="white", mec="#2a78d6")
    ax.axhline(FROZEN_S[tr][0], color="#8d8d8d", lw=1, ls=":", label="frozen BEATs LDN1, development")
    ax.set_xticks(xs, labels); ax.set_title(f"{tr} · {TRACK_INPUT[tr]} at −6 dB", loc="left", color=INK, fontsize=10.5)
    ax.grid(True, color=GRID, lw=0.6); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("score S = mean of (A + B+10)/2", color=MUTED)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", ncol=3, fontsize=9)
fig.suptitle("Notebook 21 · each ingredient against the current recipe (open circle: not kept)", fontsize=11)
fig.tight_layout(rect=(0, 0.08, 1, 0.95)); fig.savefig(OUT / "strong_finetune_steps.png", dpi=150); plt.show()

COL = {"ARR": "#eb6834", "MIC+": "#2a78d6", "MIC": "#1baf7a"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, s in zip(axes, ("A", "B+10")):
    x = np.arange(len(TYPES)); w = 0.8 / (len(TRACKS) + 1)
    for j, tr in enumerate(TRACKS):
        ax.bar(x + (j - len(TRACKS) / 2) * w, [final_res(tr)[t][s]["avg"] for t in TYPES], w * 0.92, color=COL.get(tr, "#8d8d8d"), label=f"{tr} final")
    if s == "A":
        ax.bar(x + (len(TRACKS) / 2) * w, [PUBLISHED_M6[t] for t in TYPES], w * 0.92, color="#c9c8c4", label="arXiv 2309.07500 (one microphone)")
    ax.set_xticks(x, TYPES); ax.set_ylim(0.5, 1.0); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.set_title("known machine (A)" if s == "A" else "new machine, 10 clips (B+10)", loc="left", color=INK, fontsize=10.5)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("average AUC at −6 dB (LDN1)", color=MUTED)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", ncol=4, fontsize=9)
fig.tight_layout(rect=(0, 0.1, 1, 1)); fig.savefig(OUT / "strong_finetune_final.png", dpi=150); plt.show()

ZIP = OUT / "p04_n21_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("strong_finetune.json", "strong_finetune_steps.png", "strong_finetune_final.png"):
        z.write(OUT / f, f)
print("saved", ZIP, f"· {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n21_results.zip`). It is also in Drive › `P04_StrongFineTune`.

The trained adapters (`ft_*.pt`) and embeddings (`emb_*.npz`) stay in Drive, because notebooks 22 and 23 need them.